# 06 — Incremental Load

Simulates a new period of business activity and appends it to existing tables.

Schedule this notebook in a Fabric pipeline (e.g. weekly) to keep demo data fresh.
It auto-detects the last loaded date and advances forward by `PERIOD_DAYS`.

In [ ]:
%run ./00_helpers

In [ ]:
RANDOM_SEED               = -1      # -1 = non-deterministic (uses current timestamp)
SCHEMA_NAME               = "rockline"
PERIOD_DAYS               = 7       # calendar days to simulate forward from last watermark
DAILY_ORDER_TARGET        = 45
MAX_ORDER_LINES           = 12
MAX_PO_LINES              = 15
TRADE_DISCOUNT_MAX_PCT    = 15.0
VAT_RATE                  = 0.20
NEW_CUSTOMERS             = 10      # additional customer records to append
NEW_PRODUCTS              = 0       # additional SKUs to append (0 = none)
SIMULATE_SEASONAL_WEIGHTS = "true"  # "true" or "false"

## Determine Watermark

In [ ]:
import random
from datetime import datetime, date, timedelta
from decimal import Decimal

_NOW = datetime.utcnow()
seed = RANDOM_SEED if int(RANDOM_SEED) >= 0 else int(_NOW.timestamp())
rng  = random.Random(seed)

# Find the last loaded date from sales orders
max_order_row = spark.sql(f"SELECT MAX(CAST(order_date AS DATE)) AS max_date FROM {SCHEMA_NAME}.fact_sales_order").collect()
last_loaded   = max_order_row[0]["max_date"] if max_order_row[0]["max_date"] else date.today() - timedelta(days=8)

inc_start = last_loaded + timedelta(days=1)
inc_end   = inc_start   + timedelta(days=int(PERIOD_DAYS) - 1)

print(f"Last loaded date : {last_loaded}")
print(f"Incremental window: {inc_start} to {inc_end}")

## Determine ID Offsets

In [ ]:
max_so  = spark.sql(f"SELECT COALESCE(MAX(sales_order_id), 0) AS m FROM {SCHEMA_NAME}.fact_sales_order").collect()[0]["m"]
max_po  = spark.sql(f"SELECT COALESCE(MAX(purchase_order_id), 0) AS m FROM {SCHEMA_NAME}.fact_purchase_order").collect()[0]["m"]
max_cid = spark.sql(f"SELECT COALESCE(MAX(customer_id), 0) AS m FROM {SCHEMA_NAME}.dim_customer").collect()[0]["m"]
max_pid = spark.sql(f"SELECT COALESCE(MAX(product_id), 0) AS m FROM {SCHEMA_NAME}.dim_product").collect()[0]["m"]

print(f"Max sales_order_id   : {max_so}")
print(f"Max purchase_order_id: {max_po}")
print(f"Max customer_id      : {max_cid}")
print(f"Max product_id       : {max_pid}")

## Append New Customers

In [ ]:
n_new = int(NEW_CUSTOMERS)
if n_new > 0:
    from faker import Faker
    fake = Faker("en_GB")
    Faker.seed(seed)

    _TRADE_SUFFIX = [" Construction Ltd", " Building Contractors", " Groundworks Ltd",
                     " Civil Engineering Ltd", " Property Developments Ltd"]
    branch_ids = [b["branch_id"] for b in BRANCHES]
    new_cust_rows = []
    for j in range(1, n_new + 1):
        cid = max_cid + j
        is_trade = rng.random() < 0.75
        first, last = fake.first_name(), fake.last_name()
        if is_trade:
            company = fake.company() + rng.choice(_TRADE_SUFFIX)
            new_cust_rows.append((
                cid, "trade", company, first, last,
                trade_email(company), random_uk_phone(rng),
                fake.street_address(), fake.city(), fake.postcode(),
                f"TRD-{cid:05d}",
                Decimal(str(rng.choice([5000,10000,25000]))),
                30, rng.choice(branch_ids), None,
                True, _NOW,
            ))
        else:
            new_cust_rows.append((
                cid, "private", None, first, last,
                fake.email(), random_uk_phone(rng),
                fake.street_address(), fake.city(), fake.postcode(),
                f"PVT-{cid:05d}",
                None, 0, rng.choice(branch_ids), None,
                True, _NOW,
            ))

    new_cust_df = to_spark_df(new_cust_rows, SCHEMA_DIM_CUSTOMER)
    new_cust_df.write.format("delta").mode("append").saveAsTable(f"{SCHEMA_NAME}.dim_customer")
    print(f"Appended {n_new} new customers")
else:
    print("No new customers requested")

## Run Incremental Sales & Purchasing

In [ ]:
def _args(**kwargs):
    return {k: str(v) for k, v in kwargs.items()}

print("\n── Incremental Sales ───────────────────────────────────────────────")
mssparkutils.notebook.run("03_sales", timeout_seconds=600, arguments=_args(
    RANDOM_SEED=seed,
    SCHEMA_NAME=SCHEMA_NAME,
    START_DATE=str(inc_start),
    END_DATE=str(inc_end),
    DAILY_ORDER_TARGET=DAILY_ORDER_TARGET,
    MAX_ORDER_LINES=MAX_ORDER_LINES,
    TRADE_DISCOUNT_MAX_PCT=TRADE_DISCOUNT_MAX_PCT,
    VAT_RATE=VAT_RATE,
    WRITE_MODE="append",
    ORDER_ID_OFFSET=str(max_so),
))

print("\n── Incremental Purchasing ──────────────────────────────────────────")
mssparkutils.notebook.run("04_purchasing", timeout_seconds=600, arguments=_args(
    RANDOM_SEED=seed,
    SCHEMA_NAME=SCHEMA_NAME,
    START_DATE=str(inc_start),
    END_DATE=str(inc_end),
    VAT_RATE=VAT_RATE,
    MAX_PO_LINES=MAX_PO_LINES,
    WRITE_MODE="append",
    PO_ID_OFFSET=str(max_po),
))

## Append Inventory Snapshot

In [ ]:
new_snap_df = spark.sql(f'''
    SELECT
        (SELECT COALESCE(MAX(snapshot_id), 0) FROM {SCHEMA_NAME}.fact_inventory_snapshot) + ROW_NUMBER() OVER (ORDER BY product_id, branch_id) AS snapshot_id,
        product_id,
        branch_id,
        DATE("{inc_end}") AS snapshot_date,
        quantity_on_hand,
        quantity_reserved,
        quantity_available,
        reorder_point,
        max_stock_level,
        average_cost_gbp,
        CURRENT_TIMESTAMP() AS created_at
    FROM {SCHEMA_NAME}.fact_inventory_snapshot
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM {SCHEMA_NAME}.fact_inventory_snapshot)
''')
new_snap_df.write.format("delta").mode("append").saveAsTable(f"{SCHEMA_NAME}.fact_inventory_snapshot")
print(f"Appended new inventory snapshot for {inc_end}: {new_snap_df.count()} rows")

## Summary

In [ ]:
print(f"\n── Incremental Load Summary ─────────────────────────────────────────")
print(f"  Period:    {inc_start} to {inc_end}  ({PERIOD_DAYS} days)")
print(f"  New customers: {NEW_CUSTOMERS}")

new_so  = spark.sql(f"SELECT COUNT(*) AS c FROM {SCHEMA_NAME}.fact_sales_order WHERE sales_order_id > {max_so}").collect()[0]["c"]
new_sol = spark.sql(f"SELECT COUNT(*) AS c FROM {SCHEMA_NAME}.fact_sales_order_line WHERE sales_order_id > {max_so}").collect()[0]["c"]
new_po  = spark.sql(f"SELECT COUNT(*) AS c FROM {SCHEMA_NAME}.fact_purchase_order WHERE purchase_order_id > {max_po}").collect()[0]["c"]
print(f"  New sales orders:  {new_so:,}")
print(f"  New order lines:   {new_sol:,}")
print(f"  New purchase orders: {new_po:,}")
print("Incremental load complete.")